In [1]:
from transformers import MarianMTModel, MarianTokenizer
from sentence_transformers import SentenceTransformer, util
from datasets import load_dataset, concatenate_datasets
import pandas as pd
import torch
import sacrebleu
from tqdm import tqdm

c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load translation model and tokenizer
def load_translation_model(model_name):
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name)
    return tokenizer, model

In [3]:
# Translate a sentence
def translate(texts, tokenizer, model):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = model.generate(**inputs)
    return [tokenizer.decode(t, skip_special_tokens=True) for t in outputs]

In [4]:
# Load models
cy2en_tokenizer, cy2en_model = load_translation_model("Helsinki-NLP/opus-mt-cy-en")
en2cy_tokenizer, en2cy_model = load_translation_model("Helsinki-NLP/opus-mt-en-cy")

c:\Users\c24082331\OneDrive - Cardiff University\Desktop\RA(UniversalCEFR)\development\universalcefr\Lib\site-packages\transformers\models\marian\tokenization_marian.py:177: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [5]:
# Load SentenceTransformer model
sim_model = SentenceTransformer('sentence-transformers/distiluse-base-multilingual-cased-v2')

In [6]:
sim_model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False}) with Transformer model: DistilBertModel 
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Dense({'in_features': 768, 'out_features': 512, 'bias': True, 'activation_function': 'torch.nn.modules.activation.Tanh'})
)

In [9]:
#Semantic similarity
def semantic_similarity(s1, s2):
    emb1 = sim_model.encode(s1, convert_to_tensor=True)
    emb2 = sim_model.encode(s2, convert_to_tensor=True)
    return util.cos_sim(emb1, emb2).item()

In [10]:
def evaluate(original, back_translated):
    bleu = sacrebleu.corpus_bleu([back_translated], [[original]]).score
    chrf = sacrebleu.corpus_chrf([back_translated], [[original]]).score
    cosine = semantic_similarity(original, back_translated)
    return round(bleu, 2), round(chrf, 2), round(cosine, 4)

In [12]:
# Load Welsh CEFR data
welsh_data = load_dataset("UniversalCEFR/learn_welsh_cy")["train"]

# Filter A1 and A2 level texts
welsh_a1_a2 = welsh_data.filter(lambda example: example["cefr_level"] in ["A1", "A2"])

df = welsh_a1_a2.to_pandas()[["text", "cefr_level"]].dropna().reset_index(drop=True)

Filter: 100%|██████████| 1372/1372 [00:00<00:00, 23901.45 examples/s]


In [13]:
welsh_data

Dataset({
    features: ['title', 'lang', 'source_name', 'format', 'category', 'cefr_level', 'license', 'text'],
    num_rows: 1372
})

In [14]:
df

,text,cefr_level
0,"A: Helô, Eryl dw i. Pwy dych chi?\nB: Bore da,...",A1
1,"A: O na, yr heddlu! (Stopio'r car)\nB: Hello, ...",A1
2,"A: Bore da. Sut dych chi?\nB: Iawn, ond wedi b...",A1
3,"Ceri: Noswaith dda, Eryl. Sut wyt ti?\nEryl: D...",A1
4,A: Bore da.\nB: Hmff.\nA: Sut dych chi heddiw?...,A1
...,...,...
1367,Allech chi gyrraedd yn gynnar?,A2
1368,Allet ti gyrraedd yn gynnar?,A2
1369,Allai hi gyrraedd yn gynnar?,A2
1370,Allen nhw gyrraedd yn gynnar?,A2


In [15]:
# Count A1 and A2
df["cefr_level"].value_counts()

cefr_level
A1    764
A2    608
Name: count, dtype: int64

In [16]:
records = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Back-translating"):
    original_cy = row["text"]
    cefr = row["cefr_level"]
    
    try:
        # Welsh to English
        english = translate([original_cy], cy2en_tokenizer, cy2en_model)[0]

        # English back to Welsh
        back_cy = translate([english], en2cy_tokenizer, en2cy_model)[0]

        # Evaluate
        bleu, chrf, cosine = evaluate(original_cy, back_cy)

        records.append({
            "original_welsh": original_cy,
            "translated_english": english,
            "back_translated_welsh": back_cy,
            "cefr_level": cefr,
            "BLEU": bleu,
            "chrF": chrf,
            "cosine_similarity": cosine
        })

    except Exception as e:
        print(f"Error on: {original_cy}\n{e}")

Back-translating: 100%|██████████| 1372/1372 [47:20<00:00,  2.07s/it] 


In [17]:
df_bt = pd.DataFrame(records)
filtered_df = df_bt[df_bt["cosine_similarity"] > 0.80]
filtered_df.to_csv("files/welsh_back_translation_high_similarity.csv", index=False)

In [19]:
df_bt

,original_welsh,translated_english,back_translated_welsh,cefr_level,BLEU,chrF,cosine_similarity
0,"A: Helô, Eryl dw i. Pwy dych chi?\nB: Bore da,...","And: Very, To: Where do you do?","A: Iawn, I: Lle wyt ti'n ei wneud?",A1,0.59,7.14,0.5439
1,"A: O na, yr heddlu! (Stopio'r car)\nB: Hello, ...","And: O, the police!","Ac: O, yr heddlu!",A1,0.00,3.23,0.3973
2,"A: Bore da. Sut dych chi?\nB: Iawn, ond wedi b...","And: Good morning, How do you know: That's tir...","A: bore da, Sut rwyt ti'n gwybod: Dyna, ond we...",A1,5.19,25.35,0.7434
3,"Ceri: Noswaith dda, Eryl. Sut wyt ti?\nEryl: D...","Well: Good night, though, wins. How do you fee...","Ond, nos da: Ond, sut rwyt ti'n teimlo: sut rw...",A1,0.23,9.46,0.4478
4,A: Bore da.\nB: Hmff.\nA: Sut dych chi heddiw?...,And morning: Good morning. B. Where: How do yo...,A bore: bore da. Ble rwyt ti'n gwybod heddiw? ',A1,0.00,5.08,0.4551
...,...,...,...,...,...,...,...
1367,Allech chi gyrraedd yn gynnar?,Can you get early?,A elli di gael yn gynnar?,A2,24.45,36.88,0.4879
1368,Allet ti gyrraedd yn gynnar?,Can you reach early?,A elli di gyrraedd gynnar?,A2,17.97,54.72,0.7349
1369,Allai hi gyrraedd yn gynnar?,Could she get early?,Methu cael ei alw'n gynnar?,A2,16.23,30.97,0.7885
1370,Allen nhw gyrraedd yn gynnar?,Allend them early?,Ydy pob un ohonyn nhw'n gynnar?,A2,13.13,34.06,0.6432
